# 🔒 Semana 15-16: Rendimiento, Escalabilidad y Seguridad

**Curso:** Python, SQL, Ciencia de Datos y Análisis de Negocios  
**Duración estimada:** 2 semanas  
**Nivel:** Avanzado

---

## 📋 ¿Qué vas a aprender esta semana?

| Bloque | Temas |
|--------|-------|
| ⚡ Optimización SQL | Queries lentas, EXPLAIN, índices compuestos, query hints |
| 🗃️ Caché | Caché en memoria, TTL, invalidación, patrones cache-aside |
| 🔒 Seguridad web | SQL injection, XSS, CSRF, contraseñas, JWT básico |
| 👥 Roles y permisos | RBAC, control de acceso por recurso |
| 🔍 Auditoría | Logging de eventos de seguridad, detección de anomalías |

---

> 💡 **Prerequisito:** Semanas 1-14 completadas.  
> **Instalaciones necesarias:**
> ```bash
> pip install flask sqlalchemy bcrypt
> ```

---
## ⚡ Bloque 1: Optimización de Consultas SQL

### 📘 Teoría

#### El ciclo de optimización
```
1. IDENTIFICAR queries lentas
   (logs, EXPLAIN QUERY PLAN, time.perf_counter)
        ↓
2. ENTENDER el plan de ejecución
   (¿hace full scan? ¿usa índice?)
        ↓
3. OPTIMIZAR
   (agregar índice, reescribir query, denormalizar)
        ↓
4. MEDIR de nuevo y verificar mejora
```

#### EXPLAIN QUERY PLAN en SQLite
```sql
EXPLAIN QUERY PLAN
SELECT * FROM ventas WHERE cliente_id = 42;

-- Sin índice:
-- SCAN TABLE ventas        ← lee TODA la tabla

-- Con índice:
-- SEARCH TABLE ventas USING INDEX idx_cliente (cliente_id=?)  ← directo
```

#### Tipos de índices y cuándo usarlos

| Tipo | Cuándo | Ejemplo |
|------|--------|---------|
| Simple | Filtrar por una columna frecuente | `WHERE email = ?` |
| Compuesto | Filtrar por dos columnas juntas | `WHERE depto = ? AND activo = ?` |
| Único | Garantizar unicidad + velocidad | `email UNIQUE` |

#### Anti-patrones comunes
```sql
-- ❌ Función sobre columna indexada (invalida el índice)
WHERE UPPER(nombre) = 'ANA'

-- ✅ Alternativa
WHERE nombre = 'ana' COLLATE NOCASE

-- ❌ SELECT * en tablas grandes
SELECT * FROM ventas WHERE fecha > '2024-01-01'

-- ✅ Solo las columnas necesarias
SELECT id, monto, cliente_id FROM ventas WHERE fecha > '2024-01-01'

-- ❌ IN con subquery grande
WHERE id IN (SELECT cliente_id FROM pedidos)

-- ✅ EXISTS (para muchos registros)
WHERE EXISTS (SELECT 1 FROM pedidos WHERE pedidos.cliente_id = clientes.id)
```

### 💡 Ejemplo resuelto 1 — Benchmark completo: identificar y optimizar

In [ ]:
import sqlite3
import time
import random

# ✅ Pipeline completo de optimización SQL

def crear_bd_ventas(n=200_000, indices=None):
    """Crea una BD de ventas con N registros."""
    conn = sqlite3.connect(':memory:')
    conn.execute('PRAGMA journal_mode=WAL')
    conn.execute('PRAGMA synchronous=NORMAL')

    conn.executescript("""
        CREATE TABLE clientes (
            id       INTEGER PRIMARY KEY,
            nombre   TEXT,
            email    TEXT,
            ciudad   TEXT,
            activo   INTEGER DEFAULT 1
        );
        CREATE TABLE productos (
            id        INTEGER PRIMARY KEY,
            nombre    TEXT,
            categoria TEXT,
            precio    REAL
        );
        CREATE TABLE ventas (
            id          INTEGER PRIMARY KEY,
            cliente_id  INTEGER,
            producto_id INTEGER,
            fecha       TEXT,
            cantidad    INTEGER,
            monto       REAL
        );
    """)

    ciudades   = ['Buenos Aires','Córdoba','Rosario','Mendoza','Tucumán']
    categorias = ['Electrónica','Ropa','Hogar','Deportes']

    # Insertar clientes y productos
    conn.executemany('INSERT INTO clientes VALUES (?,?,?,?,?)',
        [(i, f'Cliente_{i}', f'cli{i}@mail.com',
          random.choice(ciudades), random.randint(0,1))
         for i in range(1, 1001)])

    conn.executemany('INSERT INTO productos VALUES (?,?,?,?)',
        [(i, f'Producto_{i}', random.choice(categorias),
          round(random.uniform(10, 5000), 2))
         for i in range(1, 201)])

    # Bulk insert de ventas
    fechas = [f'2024-{m:02d}-{d:02d}' for m in range(1,13) for d in range(1,29)]
    conn.executemany('INSERT INTO ventas VALUES (?,?,?,?,?,?)',
        [(i, random.randint(1,1000), random.randint(1,200),
          random.choice(fechas), random.randint(1,10),
          round(random.uniform(10,5000),2))
         for i in range(1, n+1)])

    if indices:
        for idx in indices:
            conn.execute(idx)

    conn.commit()
    return conn

def medir(conn, query, params=(), repeticiones=5):
    """Mide el tiempo promedio de una query."""
    conn.execute(query, params).fetchall()  # warm-up
    t0 = time.perf_counter()
    for _ in range(repeticiones):
        conn.execute(query, params).fetchall()
    return (time.perf_counter() - t0) / repeticiones * 1000

# Queries de prueba
Q1 = 'SELECT * FROM ventas WHERE cliente_id = ?'
Q2 = 'SELECT cliente_id, SUM(monto) FROM ventas GROUP BY cliente_id ORDER BY SUM(monto) DESC LIMIT 10'
Q3 = 'SELECT v.*, c.nombre, p.nombre FROM ventas v JOIN clientes c ON v.cliente_id=c.id JOIN productos p ON v.producto_id=p.id WHERE c.ciudad=? AND v.fecha LIKE ?'

print('=== PASO 1: Medir SIN índices ===')
db_sin = crear_bd_ventas(indices=None)
t1_sin = medir(db_sin, Q1, (42,))
t2_sin = medir(db_sin, Q2)
t3_sin = medir(db_sin, Q3, ('Buenos Aires', '2024-06%'))
print(f'  Q1 (buscar cliente):       {t1_sin:8.2f} ms')
print(f'  Q2 (top clientes):         {t2_sin:8.2f} ms')
print(f'  Q3 (JOIN ciudad+fecha):    {t3_sin:8.2f} ms')

# Analizar plan de ejecución
print('\n=== PASO 2: EXPLAIN QUERY PLAN ===')
for row in db_sin.execute(f'EXPLAIN QUERY PLAN {Q1}', (42,)):
    print(f'  Q1: {row[3]}')
for row in db_sin.execute(f'EXPLAIN QUERY PLAN {Q3}', ('Buenos Aires','2024-06%')):
    print(f'  Q3: {row[3]}')

print('\n=== PASO 3: Agregar índices estratégicos ===')
indices = [
    'CREATE INDEX idx_ventas_cliente   ON ventas(cliente_id)',
    'CREATE INDEX idx_ventas_producto  ON ventas(producto_id)',
    'CREATE INDEX idx_ventas_fecha     ON ventas(fecha)',
    'CREATE INDEX idx_clientes_ciudad  ON clientes(ciudad)',
    # Índice compuesto para Q3
    'CREATE INDEX idx_ventas_cli_fecha ON ventas(cliente_id, fecha)',
]
db_con = crear_bd_ventas(indices=indices)

t1_con = medir(db_con, Q1, (42,))
t2_con = medir(db_con, Q2)
t3_con = medir(db_con, Q3, ('Buenos Aires', '2024-06%'))

print('\n=== PASO 4: Comparación final ===')
print(f'{'Query':30} {'Sin índice':>12} {'Con índice':>12} {'Mejora':>8}')
print('-' * 66)
for nombre, sin, con in [
    ('Q1 - buscar por cliente',  t1_sin, t1_con),
    ('Q2 - top clientes',        t2_sin, t2_con),
    ('Q3 - JOIN ciudad+fecha',   t3_sin, t3_con),
]:
    mejora = sin/con if con > 0 else float('inf')
    print(f'  {nombre:28} {sin:10.2f}ms {con:10.2f}ms {mejora:7.1f}x')

print('\n=== EXPLAIN con índices ===')
for row in db_con.execute(f'EXPLAIN QUERY PLAN {Q1}', (42,)):
    print(f'  Q1: {row[3]}')

### ✏️ Ejercicio guiado 1 — Optimizar una consulta con JOINs

Tenés una query lenta que hace JOINs entre 3 tablas. Identificá qué índices agregar.

**Pistas:**
- Las columnas en `JOIN ... ON` son las primeras candidatas a indexar
- Las columnas en `WHERE` también
- `EXPLAIN QUERY PLAN` te dice si usa 'SCAN' (malo) o 'SEARCH USING INDEX' (bueno)

In [ ]:
import sqlite3, time, random

# BD de empleados con 3 tablas
def crear_bd_empleados(con_indices=False):
    conn = sqlite3.connect(':memory:')
    conn.executescript("""
        CREATE TABLE departamentos (id INTEGER PRIMARY KEY, nombre TEXT, presupuesto REAL);
        CREATE TABLE empleados (id INTEGER PRIMARY KEY, nombre TEXT, depto_id INTEGER, salario REAL, activo INTEGER);
        CREATE TABLE proyectos (id INTEGER PRIMARY KEY, nombre TEXT, depto_id INTEGER, estado TEXT);
        CREATE TABLE asignaciones (empleado_id INTEGER, proyecto_id INTEGER, horas INTEGER);
    """)
    deptos = [(i, f'Depto_{i}', random.uniform(100000,500000)) for i in range(1, 21)]
    emps   = [(i, f'Emp_{i}', random.randint(1,20), random.uniform(40000,120000), random.randint(0,1)) for i in range(1, 5001)]
    projs  = [(i, f'Proy_{i}', random.randint(1,20), random.choice(['activo','cerrado'])) for i in range(1, 201)]
    asigs  = [(random.randint(1,5000), random.randint(1,200), random.randint(1,40)) for _ in range(15000)]
    conn.executemany('INSERT INTO departamentos VALUES (?,?,?)', deptos)
    conn.executemany('INSERT INTO empleados VALUES (?,?,?,?,?)', emps)
    conn.executemany('INSERT INTO proyectos VALUES (?,?,?,?)', projs)
    conn.executemany('INSERT INTO asignaciones VALUES (?,?,?)', asigs)
    if con_indices:
        # ✏️ Agregá los índices que consideres necesarios:
        pass
    conn.commit()
    return conn

QUERY = """
    SELECT e.nombre, d.nombre, SUM(a.horas) as total_horas
    FROM empleados e
    JOIN departamentos d  ON e.depto_id     = d.id
    JOIN asignaciones  a  ON e.id           = a.empleado_id
    JOIN proyectos     p  ON a.proyecto_id  = p.id
    WHERE e.activo = 1
      AND p.estado = 'activo'
      AND e.salario > 60000
    GROUP BY e.id, e.nombre, d.nombre
    ORDER BY total_horas DESC
    LIMIT 10
"""

# Medir sin índices
db_sin = crear_bd_empleados(con_indices=False)
t0 = time.perf_counter()
db_sin.execute(QUERY).fetchall()
t_sin = (time.perf_counter()-t0)*1000
print(f'Sin índices: {t_sin:.2f} ms')

# ✏️ Ver el plan de ejecución:
print('\nEXPLAIN QUERY PLAN:')
for row in db_sin.execute(f'EXPLAIN QUERY PLAN {QUERY}'):
    print(f'  {row[3]}')

# ✏️ Medir CON índices (completá la función crear_bd_empleados):
db_con = crear_bd_empleados(con_indices=True)
t0 = time.perf_counter()
db_con.execute(QUERY).fetchall()
t_con = (time.perf_counter()-t0)*1000
print(f'\nCon índices: {t_con:.2f} ms')
if t_con > 0:
    print(f'Mejora:      {t_sin/t_con:.1f}x')


<details>
<summary>👀 Ver solución</summary>

```python
if con_indices:
    conn.execute('CREATE INDEX idx_emp_depto   ON empleados(depto_id)')
    conn.execute('CREATE INDEX idx_emp_activo  ON empleados(activo)')
    conn.execute('CREATE INDEX idx_emp_salario ON empleados(salario)')
    conn.execute('CREATE INDEX idx_asig_emp    ON asignaciones(empleado_id)')
    conn.execute('CREATE INDEX idx_asig_proy   ON asignaciones(proyecto_id)')
    conn.execute('CREATE INDEX idx_proy_depto  ON proyectos(depto_id)')
    conn.execute('CREATE INDEX idx_proy_estado ON proyectos(estado)')
    # Índice compuesto para el filtro más selectivo
    conn.execute('CREATE INDEX idx_emp_activo_sal ON empleados(activo, salario)')
```
</details>

### 🚀 Ejercicio libre 1 — Informe de optimización

Creá una BD de e-commerce con al menos 4 tablas y 100.000 registros. Luego:

1. Diseñá **5 queries** realistas de distinta complejidad (filtros, JOINs, agregaciones)
2. Medí el tiempo de cada una **sin ningún índice**
3. Usá `EXPLAIN QUERY PLAN` para identificar cuellos de botella
4. Agregá los índices necesarios justificando cada uno
5. Medí de nuevo y presentá la **tabla comparativa**
6. Identificá si hay alguna query que NO mejora con índices y explicá por qué

In [ ]:
import sqlite3, time, random

# 🚀 Tu informe de optimización aquí:


---
## 🗃️ Bloque 2: Caché — Patrones y Estrategias

### 📘 Teoría

El **caché** guarda resultados de operaciones costosas para servirlos rápido en la próxima solicitud.

#### Patrón Cache-Aside (el más común)
```
REQUEST
   │
   ▼
¿Está en caché? ──YES──► retornar valor cacheado
   │NO
   ▼
Consultar base de datos
   │
   ▼
Guardar en caché con TTL
   │
   ▼
Retornar valor
```

#### Métricas de caché

| Métrica | Descripción | Bueno |
|---------|-------------|-------|
| **Hit rate** | % de requests servidos desde caché | > 80% |
| **Miss rate** | % que fue a la BD | < 20% |
| **TTL** | Tiempo de vida de una entrada | Depende de la frecuencia de cambio |
| **Eviction** | Política cuando el caché está lleno | LRU (más común) |

#### Cuándo NO usar caché
- Datos que cambian constantemente (precios en tiempo real)
- Datos sensibles que no deben quedar en memoria
- Cuando el costo de invalidación supera el beneficio

### 💡 Ejemplo resuelto 2 — Sistema de caché con métricas

In [ ]:
import time
import hashlib
from collections import OrderedDict
from typing import Any, Optional

# ✅ Implementación de caché LRU con TTL y métricas

class CacheLRU:
    """
    Caché LRU (Least Recently Used) con TTL y estadísticas.
    Cuando el caché está lleno, descarta el elemento menos usado recientemente.
    """

    def __init__(self, capacidad: int = 100, ttl_segundos: int = 300):
        self.capacidad    = capacidad
        self.ttl          = ttl_segundos
        self._cache       = OrderedDict()  # preserva orden de inserción/acceso
        self._stats       = {'hits': 0, 'misses': 0, 'evictions': 0, 'expirations': 0}

    def get(self, clave: str) -> Optional[Any]:
        if clave not in self._cache:
            self._stats['misses'] += 1
            return None

        valor, timestamp = self._cache[clave]
        if time.time() - timestamp > self.ttl:
            del self._cache[clave]
            self._stats['expirations'] += 1
            self._stats['misses'] += 1
            return None

        # Mover al final = más recientemente usado
        self._cache.move_to_end(clave)
        self._stats['hits'] += 1
        return valor

    def set(self, clave: str, valor: Any, ttl: int = None):
        ttl_efectivo = ttl if ttl is not None else self.ttl
        if clave in self._cache:
            self._cache.move_to_end(clave)
        self._cache[clave] = (valor, time.time())
        if len(self._cache) > self.capacidad:
            self._cache.popitem(last=False)  # eliminar el menos usado
            self._stats['evictions'] += 1

    def invalidar(self, clave: str) -> bool:
        if clave in self._cache:
            del self._cache[clave]
            return True
        return False

    def invalidar_patron(self, patron: str) -> int:
        """Invalida todas las claves que contienen el patrón."""
        claves = [k for k in self._cache if patron in k]
        for k in claves:
            del self._cache[k]
        return len(claves)

    def stats(self) -> dict:
        total    = self._stats['hits'] + self._stats['misses']
        hit_rate = self._stats['hits'] / total * 100 if total > 0 else 0
        return {
            **self._stats,
            'total_requests': total,
            'hit_rate_pct':   round(hit_rate, 1),
            'entradas_actuales': len(self._cache),
        }

    def __len__(self): return len(self._cache)


# ── Patrón Cache-Aside ──
def obtener_con_cache(cache, clave, funcion_bd, *args):
    """Patrón cache-aside: intentar caché primero, luego BD."""
    valor = cache.get(clave)
    if valor is not None:
        return valor, 'cache'
    valor = funcion_bd(*args)
    cache.set(clave, valor)
    return valor, 'bd'


# ── Simular BD lenta ──
def consulta_bd_lenta(user_id):
    time.sleep(0.05)  # simula 50ms de latencia
    return {'id': user_id, 'nombre': f'Usuario_{user_id}', 'email': f'u{user_id}@mail.com'}


# ── Demo ──
cache = CacheLRU(capacidad=50, ttl_segundos=60)

print('=== Simulación de 100 requests ===')
import random
user_ids = [random.randint(1, 20) for _ in range(100)]  # solo 20 usuarios únicos

t0 = time.perf_counter()
for uid in user_ids:
    clave = f'usuario:{uid}'
    valor, origen = obtener_con_cache(cache, clave, consulta_bd_lenta, uid)
total_tiempo = time.perf_counter() - t0

st = cache.stats()
print(f'  Requests totales:    {st["total_requests"]}')
print(f'  Cache hits:          {st["hits"]} ({st["hit_rate_pct"]}%)')
print(f'  Cache misses (BD):   {st["misses"]}')
print(f'  Evictions:           {st["evictions"]}')
print(f'  Tiempo total:        {total_tiempo*1000:.0f} ms')
print(f'  Tiempo promedio/req: {total_tiempo/100*1000:.1f} ms')

tiempo_sin_cache = 100 * 50  # 100 requests * 50ms c/u
print(f'  Sin caché sería:     ~{tiempo_sin_cache} ms')
print(f'  Ahorro:              {(1 - total_tiempo*1000/tiempo_sin_cache)*100:.1f}%')

print('\n=== Invalidación de caché ===')
cache.set('usuario:5', {'id': 5, 'nombre': 'Ana'})  # guardar
print(f'  Antes: {cache.get("usuario:5")}')
cache.invalidar('usuario:5')                         # invalidar
print(f'  Después de invalidar: {cache.get("usuario:5")}')

# Invalidar por patrón
for i in range(1, 6):
    cache.set(f'producto:{i}', {'id': i, 'nombre': f'Prod_{i}'})
n_invalidados = cache.invalidar_patron('producto:')
print(f'  Invalidados con patrón "producto:": {n_invalidados} entradas')

### ✏️ Ejercicio guiado 2 — Caché para una API Flask

Integrá el `CacheLRU` en una app Flask para cachear las respuestas de los endpoints costosos.

**Pistas:**
- La clave del caché debe incluir todos los parámetros que afectan el resultado
- Al crear/actualizar/borrar un recurso, invalidar las entradas relacionadas
- Usá un decorador `@con_cache(ttl=60)` para mantener el código limpio

In [ ]:
from flask import Flask, jsonify, request
import time

# (Requiere la clase CacheLRU del ejemplo anterior)

app_cache = Flask('cache_demo')
cache_global = CacheLRU(capacidad=200, ttl_segundos=120)

# BD simulada
productos_bd = {
    1: {'id':1,'nombre':'Laptop','precio':1200,'categoria':'Electrónica'},
    2: {'id':2,'nombre':'Mouse', 'precio':25,  'categoria':'Electrónica'},
    3: {'id':3,'nombre':'Sillón','precio':350, 'categoria':'Hogar'},
}

def consulta_lenta_productos(categoria=None):
    time.sleep(0.1)  # simula 100ms de BD
    if categoria:
        return [p for p in productos_bd.values() if p['categoria'] == categoria]
    return list(productos_bd.values())

@app_cache.route('/productos')
def listar_productos_cacheado():
    categoria = request.args.get('categoria')
    clave = f'productos:{categoria or "todos"}'

    # ✏️ Implementá el patrón cache-aside:
    # 1. Intentar obtener del caché
    # 2. Si miss: consultar BD y guardar en caché
    # 3. Retornar resultado con header indicando origen
    pass

@app_cache.route('/productos', methods=['POST'])
def crear_producto_cacheado():
    datos = request.get_json()
    nuevo_id = max(productos_bd.keys()) + 1
    productos_bd[nuevo_id] = {'id': nuevo_id, **datos}
    # ✏️ Invalidar el caché de productos:
    pass
    return jsonify(productos_bd[nuevo_id]), 201


# Probar:
import json
hj = {'Content-Type': 'application/json'}
with app_cache.test_client() as c:
    print('=== Test de caché en Flask ===')
    for i in range(3):
        t0 = time.perf_counter()
        r = c.get('/productos')
        t = (time.perf_counter()-t0)*1000
        origen = r.headers.get('X-Cache', 'unknown')
        print(f'  Request {i+1}: {t:.0f}ms [{origen}]')


<details>
<summary>👀 Ver solución</summary>

```python
@app_cache.route('/productos')
def listar_productos_cacheado():
    categoria = request.args.get('categoria')
    clave = f'productos:{categoria or "todos"}'
    cached = cache_global.get(clave)
    if cached is not None:
        resp = jsonify(cached)
        resp.headers['X-Cache'] = 'HIT'
        return resp
    resultado = consulta_lenta_productos(categoria)
    cache_global.set(clave, resultado)
    resp = jsonify(resultado)
    resp.headers['X-Cache'] = 'MISS'
    return resp

@app_cache.route('/productos', methods=['POST'])
def crear_producto_cacheado():
    datos = request.get_json()
    nuevo_id = max(productos_bd.keys()) + 1
    productos_bd[nuevo_id] = {'id': nuevo_id, **datos}
    n = cache_global.invalidar_patron('productos:')  # invalida TODO el caché de productos
    print(f'  Cache invalidado: {n} entradas')
    return jsonify(productos_bd[nuevo_id]), 201
```
</details>

### 🚀 Ejercicio libre 2 — Caché multinivel

Implementá un sistema de caché en dos niveles:

- **L1:** Caché en memoria (rápido, capacidad limitada, TTL corto: 30s)
- **L2:** Caché persistente en SQLite (más lento que L1 pero más que la BD real, TTL: 5min)

**Lógica de acceso:**
```
Request → L1 HIT? → retornar
              ↓ MISS
          L2 HIT? → guardar en L1 → retornar
              ↓ MISS
          Consultar BD → guardar en L2 → guardar en L1 → retornar
```

**Métricas a reportar:** hit rate de L1, hit rate de L2, latencia promedio para cada nivel.

In [ ]:
import sqlite3, time
from collections import OrderedDict

# 🚀 Tu caché multinivel aquí:


---
## 🔒 Bloque 3: Seguridad Web

### 📘 Teoría

#### Los ataques más comunes

**1. SQL Injection**
```python
# ❌ VULNERABLE
query = f"SELECT * FROM users WHERE email = '{email}'"
# Si email = "'; DROP TABLE users; --"
# Ejecuta: SELECT * FROM users WHERE email = ''; DROP TABLE users; --'

# ✅ SEGURO: usar parámetros
cursor.execute('SELECT * FROM users WHERE email = ?', (email,))
```

**2. Contraseñas — nunca guardar en texto plano**
```python
import hashlib, os

# ❌ NUNCA hacer esto
password_guardado = password_usuario

# ❌ Tampoco MD5 o SHA1 (reversibles con rainbow tables)
password_guardado = hashlib.md5(password.encode()).hexdigest()

# ✅ Usar bcrypt o argon2 (con salt y miles de iteraciones)
import bcrypt
password_hash = bcrypt.hashpw(password.encode(), bcrypt.gensalt(rounds=12))
```

**3. XSS (Cross-Site Scripting)**
```python
# ❌ Renderizar input del usuario directamente en HTML
return f'<p>Bienvenido {nombre}</p>'
# Si nombre = '<script>alert(1)</script>' → ejecuta JS en el browser

# ✅ Escapar siempre el input
from html import escape
return f'<p>Bienvenido {escape(nombre)}</p>'
```

**4. CSRF (Cross-Site Request Forgery)**
```
Atacante crea una página web que hace un POST a tu API
como si fuera el usuario autenticado.
Defensa: token CSRF único por sesión.
```

### 💡 Ejemplo resuelto 3 — Demostración de SQL Injection y su prevención

In [ ]:
import sqlite3
import hashlib
import os
import html
import re

# ✅ Demostración educativa de vulnerabilidades y sus defensas

# ── 1. SQL INJECTION ──
print('=== 1. SQL Injection ===')

# Crear BD de prueba
conn = sqlite3.connect(':memory:')
conn.execute('CREATE TABLE usuarios (id INTEGER PRIMARY KEY, username TEXT, email TEXT, es_admin INTEGER DEFAULT 0)')
conn.executemany('INSERT INTO usuarios VALUES (?,?,?,?)', [
    (1, 'ana',   'ana@mail.com',   0),
    (2, 'admin', 'admin@mail.com', 1),
    (3, 'luis',  'luis@mail.com',  0),
])
conn.commit()

def buscar_usuario_vulnerable(username):
    """❌ VULNERABLE a SQL injection — NUNCA hacer esto."""
    query = f"SELECT * FROM usuarios WHERE username = '{username}'"
    return conn.execute(query).fetchall()

def buscar_usuario_seguro(username):
    """✅ SEGURO — usa parámetros."""
    return conn.execute(
        'SELECT * FROM usuarios WHERE username = ?', (username,)
    ).fetchall()

# Búsqueda normal
print('  Búsqueda normal "ana":')
print(f'    Vulnerable: {buscar_usuario_vulnerable("ana")}')
print(f'    Seguro:     {buscar_usuario_seguro("ana")}')

# SQL Injection
payload_injection = "' OR '1'='1"
print(f'\n  Payload de ataque: {payload_injection!r}')
try:
    resultado_vulnerable = buscar_usuario_vulnerable(payload_injection)
    print(f'  ❌ Vulnerable: devuelve {len(resultado_vulnerable)} usuarios (TODOS)')
except Exception as e:
    print(f'  Error: {e}')

resultado_seguro = buscar_usuario_seguro(payload_injection)
print(f'  ✅ Seguro:     devuelve {len(resultado_seguro)} usuarios (ninguno)')

# ── 2. PASSWORDS ──
print('\n=== 2. Almacenamiento de Contraseñas ===')

def hashear_inseguro(password):
    """❌ MD5 sin salt — vulnerable a rainbow tables."""
    return hashlib.md5(password.encode()).hexdigest()

def hashear_seguro(password):
    """✅ SHA-256 con salt único — mucho más seguro."""
    salt = os.urandom(32)  # 32 bytes aleatorios únicos
    clave = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, 100_000)
    return salt.hex() + ':' + clave.hex()

def verificar_seguro(password, hash_guardado):
    """Verificar contraseña contra el hash guardado."""
    salt_hex, clave_hex = hash_guardado.split(':')
    salt = bytes.fromhex(salt_hex)
    clave = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, 100_000)
    return clave.hex() == clave_hex

password = 'MiPassword123!'

h_inseguro = hashear_inseguro(password)
h_seguro   = hashear_seguro(password)

print(f'  Password original: {password}')
print(f'  MD5 (inseguro):    {h_inseguro}')
print(f'    ⚠️  Siempre el mismo hash → vulnerable a rainbow tables')
print(f'  PBKDF2 (seguro):   {h_seguro[:40]}...')
print(f'    ✅ Salt único → diferente cada vez')

# Verificar la misma password dos veces → hashes distintos
h1 = hashear_seguro(password)
h2 = hashear_seguro(password)
print(f'\n  Hashear dos veces la misma pass:')
print(f'    Hash 1: {h1[:30]}...')
print(f'    Hash 2: {h2[:30]}...')
print(f'    ¿Iguales? {h1 == h2} (esperado: False — cada uno tiene su propio salt)')
print(f'    ¿Verifican? {verificar_seguro(password, h1)} y {verificar_seguro(password, h2)}')

# ── 3. XSS ──
print('\n=== 3. XSS — Cross-Site Scripting ===')

def generar_saludo_vulnerable(nombre):
    """❌ VULNERABLE — renderiza HTML sin escapar."""
    return f'<p>Bienvenido/a, {nombre}!</p>'

def generar_saludo_seguro(nombre):
    """✅ SEGURO — escapa caracteres HTML especiales."""
    return f'<p>Bienvenido/a, {html.escape(nombre)}!</p>'

xss_payload = '<script>alert("Hackeado!"); document.cookie="stolen"</script>'

print(f'  Payload XSS: {xss_payload[:50]}...')
print(f'  Vulnerable:  {generar_saludo_vulnerable(xss_payload)}')
print(f'  Seguro:      {generar_saludo_seguro(xss_payload)}')

# ── 4. Validación de inputs ──
print('\n=== 4. Validación y Sanitización ===')

def validar_email(email):
    patron = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return bool(re.match(patron, email))

def sanitizar_texto(texto, max_len=200):
    """Elimina caracteres peligrosos y limita la longitud."""
    texto = texto.strip()
    texto = re.sub(r'[<>"\']', '', texto)  # eliminar HTML/JS básico
    return texto[:max_len]

casos = [
    ('ana@gmail.com',              True),
    ('noesunmail',                 False),
    ('test@.com',                  False),
    ('valido+tag@empresa.com.ar',  True),
]
for email, esperado in casos:
    valido = validar_email(email)
    ok = '✅' if valido == esperado else '❌'
    print(f'  {ok} {email:35} → {"válido" if valido else "inválido"}')

inputs_peligrosos = [
    '<script>alert(1)</script>',
    'texto normal con acentos áéíóú',
    'SELECT * FROM users; DROP TABLE users;',
]
print('\n  Sanitización de texto:')
for inp in inputs_peligrosos:
    print(f'  Original:    {inp[:50]}')
    print(f'  Sanitizado:  {sanitizar_texto(inp)}')
    print()

### ✏️ Ejercicio guiado 3 — API segura con contraseñas hasheadas

Implementá un sistema de autenticación seguro en Flask.

**Pistas:**
- Usá `hashlib.pbkdf2_hmac` del ejemplo anterior (no `hashlib.md5`)
- Nunca retornes el `password_hash` en las respuestas JSON
- Validá el email con regex antes de registrar
- Usá parámetros `?` en todas las queries SQL

In [ ]:
from flask import Flask, request, jsonify
from sqlalchemy import create_engine, Column, Integer, String, text
from sqlalchemy.orm import DeclarativeBase, Session
import hashlib, os, re, json

class Base3(DeclarativeBase): pass

class UsuarioSeguro(Base3):
    __tablename__ = 'usuarios_seguros'
    id            = Column(Integer, primary_key=True, autoincrement=True)
    username      = Column(String(50), unique=True, nullable=False)
    email         = Column(String(100), unique=True, nullable=False)
    password_hash = Column(String(200), nullable=False)
    salt          = Column(String(64), nullable=False)

    @staticmethod
    def hashear(password):
        salt = os.urandom(32)
        clave = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, 100_000)
        return salt.hex(), clave.hex()

    def verificar(self, password):
        salt  = bytes.fromhex(self.salt)
        clave = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, 100_000)
        return clave.hex() == self.password_hash

    def to_dict(self):
        # ✅ NUNCA incluir password_hash ni salt
        return {'id': self.id, 'username': self.username, 'email': self.email}

eng3 = create_engine('sqlite:///:memory:', echo=False)
Base3.metadata.create_all(eng3)

app3 = Flask('seguro')

EMAIL_RE = re.compile(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$')

@app3.route('/registro', methods=['POST'])
def registro_seguro():
    d = request.get_json() or {}
    # ✏️ Validaciones:
    # 1. username, email y password son requeridos
    # 2. email debe tener formato válido
    # 3. password debe tener mínimo 8 caracteres
    # 4. Verificar que el username/email no existan
    # 5. Hashear el password con pbkdf2_hmac
    # 6. Guardar en BD y retornar to_dict() (SIN el hash)
    pass

@app3.route('/login', methods=['POST'])
def login_seguro():
    d = request.get_json() or {}
    # ✏️ Verificar credenciales de forma segura:
    # 1. Buscar usuario por username
    # 2. Verificar password con usuario.verificar()
    # 3. En caso de error, dar mensaje genérico (no decir si el usuario existe)
    pass


hj = {'Content-Type': 'application/json'}
with app3.test_client() as c:
    r = c.post('/registro', data=json.dumps({'username':'ana','email':'ana@mail.com','password':'Segura123!'}), headers=hj)
    print(f'Registro: {r.status_code} → {r.get_json()}')
    r = c.post('/login', data=json.dumps({'username':'ana','password':'Segura123!'}), headers=hj)
    print(f'Login correcto: {r.status_code} → {r.get_json()}')
    r = c.post('/login', data=json.dumps({'username':'ana','password':'MalPass'}), headers=hj)
    print(f'Login incorrecto: {r.status_code} → {r.get_json()}')


### 🚀 Ejercicio libre 3 — API completamente segura

Construí una API REST con todas las capas de seguridad:

1. **Contraseñas** — PBKDF2 con salt, nunca exponer el hash
2. **SQL** — solo parámetros, nunca concatenación de strings
3. **Validación** — email, longitudes mínimas/máximas, tipos
4. **Sanitización** — escapar HTML en todos los campos de texto
5. **Rate limiting básico** — máximo 5 intentos de login fallidos por IP en 1 minuto
6. **Mensajes de error seguros** — nunca revelar si el usuario existe o no

Incluí tests que verifiquen que cada vulnerabilidad está correctamente mitigada.

In [ ]:
from flask import Flask, request, jsonify
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.orm import DeclarativeBase, Session
import hashlib, os, re, json, time
from collections import defaultdict

# 🚀 Tu API segura aquí:


---
## 👥 Bloque 4: Roles, Permisos y Auditoría

### 📘 Teoría

#### RBAC — Role-Based Access Control
Asignás permisos a **roles**, no directamente a usuarios. Los usuarios tienen roles.

```
Usuario → tiene → Rol(es) → tienen → Permisos

Ejemplo:
  ana       → [vendedor]         → [leer_productos, crear_pedido]
  carlos    → [admin]            → [todo]
  bot_sync  → [lector_api]       → [leer_productos]
```

#### Principio de mínimo privilegio
> Cada usuario/proceso debe tener **solo los permisos que necesita** para su función.  
> Si un vendedor solo necesita crear pedidos, no debe poder borrar productos.

#### Auditoría — qué registrar
```
QUIÉN  → usuario_id o IP
QUÉ    → acción (login, crear, modificar, borrar)
CUÁNDO → timestamp
DÓNDE  → endpoint, IP
RESULTADO → éxito o fracaso + mensaje
```

### 💡 Ejemplo resuelto 4 — RBAC y log de auditoría

In [ ]:
from flask import Flask, request, jsonify
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey, Boolean, text
from sqlalchemy.orm import DeclarativeBase, relationship, Session
from datetime import datetime
import json, hashlib

# ✅ Sistema RBAC con auditoría

class Base4(DeclarativeBase): pass

class Rol(Base4):
    __tablename__ = 'roles'
    id       = Column(Integer, primary_key=True, autoincrement=True)
    nombre   = Column(String(50), unique=True, nullable=False)
    permisos = Column(String(500), default='')  # CSV de permisos
    usuarios = relationship('UsuarioRBAC', back_populates='rol')

    def tiene_permiso(self, permiso):
        return permiso in self.permisos.split(',') or 'admin' in self.permisos.split(',')

class UsuarioRBAC(Base4):
    __tablename__ = 'usuarios_rbac'
    id            = Column(Integer, primary_key=True, autoincrement=True)
    username      = Column(String(50), unique=True)
    password_hash = Column(String(64))
    rol_id        = Column(Integer, ForeignKey('roles.id'))
    activo        = Column(Boolean, default=True)
    rol           = relationship('Rol', back_populates='usuarios')

    @staticmethod
    def hashear(p): return hashlib.sha256(p.encode()).hexdigest()
    def verificar(self, p): return self.password_hash == self.hashear(p)

class LogAuditoria(Base4):
    __tablename__ = 'log_auditoria'
    id         = Column(Integer, primary_key=True, autoincrement=True)
    timestamp  = Column(String, default=lambda: datetime.now().isoformat())
    usuario    = Column(String(50))
    accion     = Column(String(100))
    recurso    = Column(String(200))
    resultado  = Column(String(10))  # 'ok' o 'error'
    detalle    = Column(String(500))
    ip         = Column(String(45))

eng4 = create_engine('sqlite:///:memory:', echo=False)
Base4.metadata.create_all(eng4)

# Crear roles y usuarios
with Session(eng4) as s:
    roles = {
        'admin':    Rol(nombre='admin',    permisos='admin'),
        'vendedor': Rol(nombre='vendedor', permisos='leer_productos,crear_pedido,ver_pedidos'),
        'cliente':  Rol(nombre='cliente',  permisos='leer_productos,crear_pedido'),
    }
    s.add_all(roles.values())
    s.flush()
    usuarios = [
        UsuarioRBAC(username='super_admin', password_hash=UsuarioRBAC.hashear('admin123'), rol_id=roles['admin'].id),
        UsuarioRBAC(username='vendedor_ana',password_hash=UsuarioRBAC.hashear('vend456'),  rol_id=roles['vendedor'].id),
        UsuarioRBAC(username='cliente_luis',password_hash=UsuarioRBAC.hashear('cli789'),   rol_id=roles['cliente'].id),
    ]
    s.add_all(usuarios)
    s.commit()

app4 = Flask('rbac')

def autenticar_rbac(req):
    u, p = req.headers.get('X-User'), req.headers.get('X-Pass')
    if not u or not p: return None
    with Session(eng4) as s:
        usuario = s.query(UsuarioRBAC).filter_by(username=u, activo=True).first()
        if usuario and usuario.verificar(p):
            s.expunge(usuario)
            s.expunge(usuario.rol)
            return usuario
    return None

def requerir_permiso(permiso):
    """Decorador que verifica permisos y registra en auditoría."""
    from functools import wraps
    def decorador(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            usuario = autenticar_rbac(request)
            ip = request.remote_addr or '127.0.0.1'

            if not usuario:
                _auditar(None, f'acceso:{permiso}', request.path, 'error', 'No autenticado', ip)
                return jsonify({'error': 'No autorizado'}), 401

            if not usuario.rol.tiene_permiso(permiso):
                _auditar(usuario.username, f'acceso:{permiso}', request.path, 'error',
                         f'Permiso denegado (rol: {usuario.rol.nombre})', ip)
                return jsonify({'error': 'Permiso denegado', 'rol': usuario.rol.nombre,
                                'permiso_requerido': permiso}), 403

            _auditar(usuario.username, f'acceso:{permiso}', request.path, 'ok', '', ip)
            return func(usuario, *args, **kwargs)
        return wrapper
    return decorador

def _auditar(usuario, accion, recurso, resultado, detalle, ip):
    with Session(eng4) as s:
        s.add(LogAuditoria(usuario=usuario or 'anónimo', accion=accion,
                           recurso=recurso, resultado=resultado,
                           detalle=detalle, ip=ip))
        s.commit()

@app4.route('/productos', methods=['GET'])
@requerir_permiso('leer_productos')
def listar(usuario):
    return jsonify({'productos': ['Laptop', 'Mouse'], 'accedido_por': usuario.username})

@app4.route('/admin/usuarios', methods=['GET'])
@requerir_permiso('admin')
def admin_usuarios(usuario):
    with Session(eng4) as s:
        users = s.query(UsuarioRBAC).all()
        return jsonify([{'username': u.username, 'rol': u.rol.nombre} for u in users])

@app4.route('/auditoria', methods=['GET'])
@requerir_permiso('admin')
def ver_auditoria(usuario):
    with Session(eng4) as s:
        logs = s.query(LogAuditoria).order_by(LogAuditoria.id.desc()).limit(20).all()
        return jsonify([{'timestamp': l.timestamp, 'usuario': l.usuario,
                         'accion': l.accion, 'resultado': l.resultado,
                         'detalle': l.detalle} for l in logs])

# ── Pruebas ──
print('=== Pruebas de RBAC y Auditoría ===')
hj = {'Content-Type': 'application/json'}
auth = {
    'admin':    {**hj, 'X-User': 'super_admin',  'X-Pass': 'admin123'},
    'vendedor': {**hj, 'X-User': 'vendedor_ana',  'X-Pass': 'vend456'},
    'cliente':  {**hj, 'X-User': 'cliente_luis',  'X-Pass': 'cli789'},
}

with app4.test_client() as c:
    # Accesos
    print('\nAcceso a /productos:')
    for rol, headers in auth.items():
        r = c.get('/productos', headers=headers)
        print(f'  {rol:10} → {r.status_code}')

    print('\nAcceso a /admin/usuarios:')
    for rol, headers in auth.items():
        r = c.get('/admin/usuarios', headers=headers)
        print(f'  {rol:10} → {r.status_code}: {r.get_json().get("error", "OK")}')

    # Sin autenticación
    r = c.get('/productos')
    print(f'\nSin auth → {r.status_code}')

    # Ver log de auditoría
    r = c.get('/auditoria', headers=auth['admin'])
    print('\n=== Log de Auditoría ===')
    for entry in r.get_json():
        icon = '✅' if entry['resultado'] == 'ok' else '❌'
        print(f'  {icon} {entry["usuario"]:15} | {entry["accion"]:30} | {entry["detalle"] or ""}')

### 🚀 Ejercicio libre 4 — Sistema de seguridad completo

Integrá todos los conceptos de seguridad en una sola app:

**Funcionalidades:**
1. **Registro seguro** — PBKDF2, validación de email y contraseña fuerte
2. **Login con rate limiting** — máximo 5 intentos fallidos por usuario en 15 minutos, luego lockout
3. **RBAC con 3 roles** — `admin`, `editor`, `lector`
4. **Auditoría completa** — cada acceso (exitoso o fallido) queda registrado
5. **Endpoint de seguridad** — `GET /admin/seguridad/report` que muestra:
   - Usuarios con más intentos fallidos
   - IPs con más accesos denegados
   - Últimos 10 eventos de seguridad

**Tests requeridos:**
- Login correcto → 200
- 6 intentos fallidos → lockout (429)
- Acceso sin permiso → 403 con log en auditoría
- El log de auditoría contiene los eventos correctos

In [ ]:
from flask import Flask, request, jsonify
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey, Boolean
from sqlalchemy.orm import DeclarativeBase, relationship, Session
import hashlib, os, json, time
from collections import defaultdict
from datetime import datetime

# 🚀 Tu sistema de seguridad completo aquí:


---
## ✅ Resumen de la Semana 15-16

### Lo que aprendiste

| Tema | Conceptos clave |
|------|-----------------|
| Optimización SQL | EXPLAIN, índices simples y compuestos, anti-patrones |
| Caché LRU | TTL, hit/miss rate, patrón cache-aside, invalidación |
| Seguridad | SQL injection, XSS, PBKDF2, validación y sanitización |
| RBAC | Roles, permisos, principio de mínimo privilegio |
| Auditoría | Log de eventos, quién/qué/cuándo/resultado |

### ➡️ Próximos pasos — Semana 17-18
- Patrones de diseño: MVC, Factory, Observer, Strategy
- Fundamentos de ciencia de datos: EDA, limpieza, modelado predictivo

---

### 📚 Recursos recomendados
- [OWASP Top 10](https://owasp.org/www-project-top-ten/) — las 10 vulnerabilidades más críticas
- [Use The Index, Luke!](https://use-the-index-luke.com/) — guía profunda de índices SQL
- [Have I Been Pwned](https://haveibeenpwned.com/) — por qué importan las contraseñas seguras
- [Python Security Best Practices](https://snyk.io/blog/python-security-best-practices-cheat-sheet/)